## Not Defteri 4: Soft Mask'ın Model Karar Mekanizmasına ve Performansına Etkisi

#### Amaç

`step_02`'te üretilen akciğer-odaklı (soft mask) veri ile ham veri, **aynı mimari, aynı tohumlar ve aynı eğitim protokolüyle** karşılaştırılır. Amaç yalnızca "hangisinin AUC'si yüksek" sorusunu yanıtlamak değil, **modelin kararını nereye dayandırdığını** (akciğer içi mi, dışı mı) ölçmektir. Bu defter bir *yöntem karşılaştırmasıdır*, klinik tanı aracı değildir.

> **Bu defterde sonuç yoktur; sonuçları çalıştırdığınızda üretir.** Aşağıdaki hipotezler ve kabul ölçütleri, sonuçlar görülmeden **önce** tanımlanmıştır (ön-kayıt mantığı); sonuçlar beklentiye uymazsa bu da raporlanır.

#### Deney Tasarımı

| Model | Girdi verisi | Eğitim girdisi | Amacı |
|---|---|---|---|
| **A_raw** | ham | tam görüntü | Referans (baseline) |
| **B_soft** | soft mask (+fallback) | tam görüntü | Önerilen ön işleme |
| **C_outside** | ham | **yalnızca akciğer dışı** (akciğer sabit gri) | Akciğer dışında pnömoniyle ilişkili sinyal var mı? (H1) |
| **D_lung** | ham | **yalnızca akciğer içi** (dış sabit gri) | Akciğer içindeki sinyalin üst referansı |

Tüm modeller aynı mimari (ResNet-18, tek kanallı giriş), aynı kayıp (BCE, sınıf ağırlıklı), aynı optimizasyon ve aynı erken durdurma kuralıyla eğitilir. Model seçimi **doğrulama (val)** AUPRC'sine göre yapılır; **test kümesi hiçbir model/hiperparametre/eşik seçiminde kullanılmaz.**

#### Hipotezler ve Önceden Tanımlı Kabul Ölçütleri

| Kod | Hipotez | Ölçüt (çalıştırmadan önce belirlendi) |
|---|---|---|
| **H1** | Akciğer dışında pnömoniyle ilişkili sinyal vardır. | C_outside AUROC'unun %95 güven aralığı alt sınırı **> 0,55** → *desteklendi*; alt sınır 0,50–0,55 → *zayıf kanıt*; ≤ 0,50 → *desteklenmedi*. |
| **H2** | B, A'ya göre akciğer dışı bilgiye daha az bağımlıdır. | R_dış = AUROC(tam) − AUROC(yalnızca-akciğer girdisi). ΔR = R_dış(B) − R_dış(A) için %95 GA üst sınırı **< 0** → *desteklendi*. |
| **H3** | B'nin Grad-CAM dikkati akciğer içinde daha yoğundur. | Akciğer içi dikkat oranı farkı (B − A), eşleştirilmiş Wilcoxon p < 0,05 ve medyan fark > 0 → *desteklendi*. |
| **H4** | Soft mask genel performansı anlamlı düşürmez (aşağı olmama). | ΔAUROC = AUROC(B) − AUROC(A) için %95 GA alt sınırı **> −0,02** → *aşağı olmama sağlandı*. (−0,02 payı, klinik/pratik olarak küçük kabul edilen keyfi bir değerdir.) |

#### Bilinen Sınırlılıklar (tasarım aşamasında)

1. **Tek veri kaynağı.** Hastane/cihaz dışı doğrulama yok; "genelleme" iddiası yapılamaz, kısayolun azaldığı yalnızca **dolaylı** göstergelerle (H1–H3) incelenir.
2. **H2 kısmen döngüsel.** B'nin dış bölgesi zaten ×0,25 sönük olduğundan, dışı sıfırlamanın B üzerindeki etkisinin küçük çıkması yapıdan da kaynaklanır. Bu yüzden H2, H1 (C modeli) ve H3 ile **birlikte** yorumlanır.
3. **Maskeleme girdi kaymasına yol açar.** Ablasyon girdilerinde (akciğer ya da dış sabit griye çekilir) modelin eğitimde görmediği görüntüler oluşur; bu, performans düşüşünü kısayoldan bağımsız olarak da artırabilir.
4. **Güven aralıkları yalnızca test örneklemesini kapsar** (bootstrap); tohum varyansı ayrıca ± SS olarak verilir. C ve D tek tohumla eğitilir (süre kısıtı).
5. **Test'te ≈ 242 pozitif.** Küçük farklar anlamlı değildir; AUPRC özellikle gürültülüdür (rastgele taban ≈ sınıf oranı ≈ 0,011).
6. **Etiket gürültüsü ve çözünürlük (128×128)** performansın üst sınırını belirler (bkz. `step_02`).

#### 1. Kurulum, Çalışma Zamanı ve Yapılandırma

In [ ]:
import os, json, time, copy, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from scipy.stats import wilcoxon

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Colab dışında çalışılıyor; dosya yollarını kontrol edin.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("UYARI: GPU yok. Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçin (TPU bu kodla çalışmaz).")

CFG = dict(
    data_dir = '/content/drive/MyDrive/Machine Learning/Datasets/MedMNIST/',
    raw_file = 'chestmnist_128.npz',
    soft_file = 'chestmnist_128_softmask_v2.npz',      # step_02'ün çıktısı (maske + bayrak içerir)
    out_dir = '/content/drive/MyDrive/Machine Learning/Datasets/MedMNIST/results_step04/',
    pneumonia_idx = 6,
    seeds = [0, 1, 2],          # A ve B için tohumlar
    aux_seeds = [0],            # C ve D için tohumlar (süre kısıtı)
    epochs = 8, batch = 128, lr = 3e-4, wd = 1e-4, patience = 3,
    arch = 'resnet18',          # 'resnet18' | 'tiny' (yalnızca hızlı duman testi için)
    pretrained = True,          # ImageNet ağırlıkları (indirilemezse rastgele başlatılır)
    pos_weight_mode = 'sqrt',   # 'sqrt' (kararlı) | 'full' (nötr/poz oranı) | 'none'
    n_boot = 1000, n_cam = 200,
    quick = False,              # True: küçük alt küme + 2 epoch (yalnızca kodu denemek için; sonuçlar raporlanmaz)
)
os.makedirs(CFG['out_dir'], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
SPLITS = ["train", "val", "test"]

#### 2. Veri Yükleme ve Bütünlük Kontrolleri

Ham (`chestmnist_128.npz`) ve soft maskeli (`..._softmask_v2.npz`) dosyalar yüklenir. **Etiketlerin iki dosyada birebir aynı olduğu**, tüm görüntüler için akciğer maskesinin üretildiği ve eksik anahtar olmadığı doğrulanır; aksi halde defter bilerek hata verir (sessizce yanlış deney yapmaktansa durmak daha güvenlidir).

In [ ]:
def _load(name):
    with np.load(os.path.join(CFG['data_dir'], name)) as z:
        return {k: z[k] for k in z.files}

raw, soft = _load(CFG['raw_file']), _load(CFG['soft_file'])
need = [f"{s}_{k}" for s in SPLITS for k in ("images", "labels", "lung", "fail")]
missing = [k for k in need if k not in soft]
if missing:
    raise KeyError(f"{CFG['soft_file']} içinde eksik anahtarlar: {missing}. step_02'ün son hücresini (v2 kaydı) çalıştırın.")

Y, XR, XS, LUNG, FAIL = {}, {}, {}, {}, {}
for s in SPLITS:
    assert np.array_equal(raw[f"{s}_labels"], soft[f"{s}_labels"]), f"{s}: etiketler iki dosyada farklı!"
    assert raw[f"{s}_images"].shape == soft[f"{s}_images"].shape
    Y[s]    = raw[f"{s}_labels"][:, CFG['pneumonia_idx']].astype(np.float32)
    XR[s]   = torch.from_numpy(raw[f"{s}_images"])
    XS[s]   = torch.from_numpy(soft[f"{s}_images"])
    LUNG[s] = torch.from_numpy(soft[f"{s}_lung"])
    FAIL[s] = soft[f"{s}_fail"].astype(bool)
    cover = (soft[f"{s}_lung"].reshape(len(Y[s]), -1).sum(1) > 0).mean()
    print(f"{s}: N={len(Y[s])} | pnömoni={int(Y[s].sum())} ({Y[s].mean():.2%}) | maske üretilen görüntü oranı={cover:.2%} | fallback={FAIL[s].mean():.2%}")
    if cover < 0.999:
        print(f"  UYARI: {s} kümesinde maskesi olmayan görüntüler var (step_02 kısmi çalışmış olabilir).")
del raw, soft

if CFG['quick']:                                    # yalnızca kodu denemek için; sonuç raporlamayın
    CFG['epochs'] = 2
    rng = np.random.default_rng(0)
    for s, n in [("train", 8000), ("val", 2000), ("test", 4000)]:
        pos = np.flatnonzero(Y[s] == 1); neg = np.flatnonzero(Y[s] == 0)
        idx = np.sort(np.concatenate([pos[: n // 10], rng.choice(neg, min(n - min(len(pos), n // 10), len(neg)), replace=False)]))
        Y[s], XR[s], XS[s], LUNG[s], FAIL[s] = Y[s][idx], XR[s][idx], XS[s][idx], LUNG[s][idx], FAIL[s][idx]
    print("QUICK MOD aktif:", {s: len(Y[s]) for s in SPLITS})

#### 3. Girdi Hazırlama, Veri Artırma, Model ve Eğitim Altyapısı

- **Girdi bileşimi (`compose`)**: `full` (tam görüntü), `lung_only` (akciğer dışı sabit griye çekilir), `outside_only` (akciğer içi sabit griye çekilir). Sabit gri, ilgili verinin eğitim ortalamasıdır.
- **Veri artırma (yalnızca eğitimde, GPU'da):** küçük döndürme (±7°), ölçek (±%8), kaydırma (±%6), parlaklık/kontrast oynatma. Yatay çevirme **kullanılmaz** (kalp/organ yerleşimi anlam taşır).
- **Model:** ResNet-18, tek kanallı giriş (ilk evrişim ağırlıkları kanallar üzerinden toplanır), tek çıkış (pnömoni lojiti).
- **Kayıp:** BCE + `pos_weight` (varsayılan √(negatif/pozitif) ≈ 9; tam oran ≈ 80 kararsızlık yaratabilir). AUROC/AUPRC kalibrasyondan bağımsız olduğundan bu seçim sıralama metriklerini yapısal olarak bozmaz.
- **Seçim:** her epoch'ta val AUPRC; en iyi ağırlıklar saklanır; `patience` epoch iyileşme yoksa durulur.

In [ ]:
# ---------- girdi bileşimi ----------
def compose(xb_u8, lb_u8, mode, fill):
    """xb_u8, lb_u8: (B,H,W) uint8 (cihazda). Dönüş: (B,1,H,W) float [0,1]."""
    x = xb_u8.float() / 255.0
    if mode == "lung_only":
        l = lb_u8.float(); x = x * l + fill * (1 - l)
    elif mode == "outside_only":
        l = lb_u8.float(); x = x * (1 - l) + fill * l
    elif mode != "full":
        raise ValueError(mode)
    return x[:, None]

def augment(x):
    B = x.shape[0]; dev = x.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(7)
    sc  = 1 + (torch.rand(B, device=dev) * 2 - 1) * 0.08
    tx  = (torch.rand(B, device=dev) * 2 - 1) * 0.06 * 2
    ty  = (torch.rand(B, device=dev) * 2 - 1) * 0.06 * 2
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = torch.cos(ang) * sc; theta[:, 0, 1] = -torch.sin(ang) * sc; theta[:, 0, 2] = tx
    theta[:, 1, 0] = torch.sin(ang) * sc; theta[:, 1, 1] =  torch.cos(ang) * sc; theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="border", align_corners=False)
    x = x * (1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.15) + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.05
    return x.clamp(0, 1)

def comp_stats(X, L, mode, fill, n=4000):
    xb = compose(X[:n], L[:n], mode, fill)
    return float(xb.mean()), float(xb.std() + 1e-6)

# ---------- model ----------
class TinyCNN(nn.Module):               # yalnızca duman testi için
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 16, 3, 2, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.Conv2d(16, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU())
        self.last_conv = nn.Conv2d(32, 64, 3, 2, 1)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = F.relu(self.last_conv(self.f(x)))
        return self.fc(x.mean((2, 3)))

def build_model():
    if CFG['arch'] == 'tiny':
        return TinyCNN()
    import torchvision
    try:
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if CFG['pretrained'] else None
        m = torchvision.models.resnet18(weights=w)
    except Exception as e:
        print("Ön-eğitimli ağırlıklar indirilemedi, rastgele başlatılıyor:", e)
        m = torchvision.models.resnet18(weights=None)
    old = m.conv1
    m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    m.conv1.weight.data = old.weight.data.sum(1, keepdim=True).clone()
    m.fc = nn.Linear(512, 1)
    return m

def cam_layer(model):
    return model.layer4 if hasattr(model, "layer4") else model.last_conv

# ---------- tahmin ----------
@torch.no_grad()
def predict(model, X, L, mode, norm, bs=512):
    fill, mean, std = norm
    model.to(device).eval(); out = []
    for i in range(0, len(X), bs):
        x = compose(X[i:i+bs].to(device), L[i:i+bs].to(device), mode, fill)
        x = (x - mean) / std
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            out.append(model(x).float().squeeze(1).cpu())
    return torch.cat(out).numpy()

def sig(z): return 1.0 / (1.0 + np.exp(-z))

# ---------- eğitim ----------
EXPS = {
    "A_raw":     dict(var="raw",  mode="full"),
    "B_soft":    dict(var="soft", mode="full"),
    "C_outside": dict(var="raw",  mode="outside_only"),
    "D_lung":    dict(var="raw",  mode="lung_only"),
}
def data_of(var, split): return XR[split] if var == "raw" else XS[split]

def train_one(name, seed):
    spec = EXPS[name]; var, mode = spec["var"], spec["mode"]
    set_seed(seed)
    Xtr, Xva = data_of(var, "train"), data_of(var, "val")
    fill = float(Xtr[:4000].float().mean() / 255.0)
    mean, std = comp_stats(Xtr, LUNG["train"], mode, fill)
    model = build_model().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    N, bs = len(Xtr), CFG['batch']
    steps = CFG['epochs'] * math.ceil(N / bs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    use_amp = (device == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ytr = torch.from_numpy(Y["train"])
    npos = float(ytr.sum()); nneg = N - npos
    pw = {"sqrt": math.sqrt(nneg / npos), "full": nneg / npos, "none": 1.0}[CFG['pos_weight_mode']]
    pw = torch.tensor(pw, device=device)
    best, best_state, bad, hist = -1.0, None, 0, []
    t0 = time.time()
    for ep in range(CFG['epochs']):
        model.train(); perm = torch.randperm(N); tot = 0.0
        for k in range(0, N, bs):
            idx = perm[k:k+bs]
            x = compose(Xtr[idx].to(device), LUNG["train"][idx].to(device), mode, fill)
            x = (augment(x) - mean) / std
            yb = ytr[idx].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logit = model(x).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(logit.float(), yb, pos_weight=pw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss) * len(idx)
        pv = predict(model, Xva, LUNG["val"], mode, (fill, mean, std))
        auroc, auprc = roc_auc_score(Y["val"], pv), average_precision_score(Y["val"], pv)
        hist.append(dict(epoch=ep + 1, loss=tot / N, val_auroc=auroc, val_auprc=auprc))
        print(f"[{name} s{seed}] ep{ep+1}/{CFG['epochs']} loss={tot/N:.4f} val AUROC={auroc:.4f} AUPRC={auprc:.4f} ({time.time()-t0:.0f}s)")
        if auprc > best:
            best, bad = auprc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG['patience']: print("  erken durdu"); break
    model.load_state_dict(best_state)
    return model.cpu(), (fill, mean, std), hist

#### 4. Eğitim

A ve B her tohum için eğitilir. C ve D yalnızca `aux_seeds` ile eğitilir. Her modelin tahminleri hem kendi girdi bileşiminde hem de (A ve B için) **ablasyon girdilerinde** test kümesinde saklanır. **Süre notu:** toplam süre GPU modeline ve epoch sayısına bağlıdır; bu defterde ölçülmedi. Önce `CFG['quick']=True` ile kodun uçtan uca çalıştığını doğrulayın (bu sonuçlar raporlanmaz).

In [ ]:
MODELS, NORMS, HISTS, PRED, TIMES = {}, {}, {}, {}, {}
plan = [(n, s) for n in ["A_raw", "B_soft"] for s in CFG['seeds']] + \
       [(n, s) for n in ["C_outside", "D_lung"] for s in CFG['aux_seeds']]

for name, seed in plan:
    t0 = time.time()
    model, norm, hist = train_one(name, seed)
    MODELS[(name, seed)], NORMS[(name, seed)], HISTS[(name, seed)] = model, norm, hist
    var, own = EXPS[name]["var"], EXPS[name]["mode"]
    modes = ["full", "outside_only", "lung_only"] if name in ("A_raw", "B_soft") else [own]
    for md_ in modes:
        PRED[(name, seed, md_, "test")] = predict(model, data_of(var, "test"), LUNG["test"], md_, norm)
    PRED[(name, seed, own, "val")] = predict(model, data_of(var, "val"), LUNG["val"], own, norm)
    TIMES[(name, seed)] = time.time() - t0
    model.cpu(); torch.cuda.empty_cache() if device == "cuda" else None

os.makedirs(CFG['out_dir'], exist_ok=True)
torch.save({f"{k[0]}_s{k[1]}": v.state_dict() for k, v in MODELS.items()}, os.path.join(CFG['out_dir'], "weights.pt"))
print("Eğitim tamam. Toplam süre (dk):", round(sum(TIMES.values()) / 60, 1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for (name, seed), h in HISTS.items():
    ls = "-" if name == "A_raw" else ("--" if name == "B_soft" else ":")
    axes[0].plot([e["loss"] for e in h], ls, label=f"{name} s{seed}")
    axes[1].plot([e["val_auprc"] for e in h], ls, label=f"{name} s{seed}")
axes[0].set_title("Eğitim kaybı"); axes[1].set_title("Val AUPRC"); axes[0].set_xlabel("epoch"); axes[1].set_xlabel("epoch")
axes[1].legend(fontsize=6); plt.tight_layout(); plt.savefig(os.path.join(CFG['out_dir'], "egitim_egrileri.png"), dpi=150); plt.show()

**Okuma notu.** Eğitim kaybı düşerken val AUPRC erken platoya girip düşüyorsa model aşırı uyum yapıyordur (nadir pozitif sınıfta sık görülür). En iyi epoch val'e göre seçildiği için test sonuçları bundan korunur, ancak val AUPRC gürültülüdür (≈134 pozitif); epoch seçimi de gürültülü olabilir.

#### 5. Genel Test Performansı (A vs B)

Her model için **tohumlar arası ortalama ± SS** verilir. Ardından aynı tohumların olasılıkları ortalanarak **topluluk (ensemble)** elde edilir; bootstrap güven aralıkları ve eşleştirilmiş karşılaştırmalar topluluk üzerinden hesaplanır. Eşik, **val kümesinde F1'i maksimize eden** değerdir (test'e bakılmaz). Doğruluk (accuracy) raporlanmaz, çünkü sınıf oranı ≈ %1 iken yanıltıcıdır.

In [ ]:
def ens(name, mode, split="test"):
    seeds = [s for s in (CFG['seeds'] if name in ("A_raw", "B_soft") else CFG['aux_seeds']) if (name, s, mode, split) in PRED]
    return np.mean([sig(PRED[(name, s, mode, split)]) for s in seeds], axis=0), seeds

def best_f1_thr(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-12)
    return float(th[int(np.argmax(f1))])

def thr_metrics(y, p, thr):
    pred = p >= thr
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    return dict(sens=sens, spec=spec, prec=prec, f1=2 * prec * sens / max(prec + sens, 1e-12), tp=tp, fp=fp, fn=fn, tn=tn)

yte = Y["test"]
RESULTS = {"config": {k: v for k, v in CFG.items()}, "n_test": int(len(yte)), "n_test_pos": int(yte.sum())}
print(f"test: N={len(yte)}, pozitif={int(yte.sum())}, taban AUPRC≈{yte.mean():.4f}\n")
for name in ["A_raw", "B_soft"]:
    per = [(roc_auc_score(yte, PRED[(name, s, "full", "test")]), average_precision_score(yte, PRED[(name, s, "full", "test")])) for s in CFG['seeds']]
    per = np.array(per)
    pe, _ = ens(name, "full"); pv, _ = ens(name, "full", "val")
    thr = best_f1_thr(Y["val"], pv); tm = thr_metrics(yte, pe, thr)
    RESULTS[name] = dict(auroc_mean=per[:, 0].mean(), auroc_sd=per[:, 0].std(ddof=1) if len(per) > 1 else float("nan"),
                         auprc_mean=per[:, 1].mean(), auprc_sd=per[:, 1].std(ddof=1) if len(per) > 1 else float("nan"),
                         auroc_ens=roc_auc_score(yte, pe), auprc_ens=average_precision_score(yte, pe), thr=thr, **tm)
    r = RESULTS[name]
    print(f"{name}: AUROC {r['auroc_mean']:.4f}±{r['auroc_sd']:.4f} | AUPRC {r['auprc_mean']:.4f}±{r['auprc_sd']:.4f} "
          f"| ens AUROC {r['auroc_ens']:.4f} AUPRC {r['auprc_ens']:.4f} | eşik={thr:.3f}: duyarlılık={r['sens']:.3f} özgüllük={r['spec']:.3f} kesinlik={r['prec']:.3f} F1={r['f1']:.3f}")

#### 6. Güven Aralıkları ve Hipotez Testleri (H1, H2, H4)

Aşağıdaki hücre test örneklemesini yeniden örnekler (bootstrap, eşleştirilmiş: tüm modeller **aynı** örneklerle değerlendirilir). Ablasyon girdileri: `lung_only` girdisi = dış bölge sabit gri (akciğer korunur); `outside_only` girdisi = akciğer sabit gri (dış korunur). **Bu testlerin uyarısı:** model bu girdileri eğitimde görmemiştir; düşüşün bir kısmı dağılım kaymasından gelir.

In [ ]:
keys = {
    "A_full": ("A_raw", "full"), "A_lung": ("A_raw", "lung_only"), "A_out": ("A_raw", "outside_only"),
    "B_full": ("B_soft", "full"), "B_lung": ("B_soft", "lung_only"), "B_out": ("B_soft", "outside_only"),
    "C": ("C_outside", "outside_only"), "D": ("D_lung", "lung_only"),
}
P = {k: ens(n, m)[0] for k, (n, m) in keys.items()}

def stats(yt, Pb):
    auc = {k: roc_auc_score(yt, v) for k, v in Pb.items()}
    out = dict(auc)
    out["ap_A"] = average_precision_score(yt, Pb["A_full"]); out["ap_B"] = average_precision_score(yt, Pb["B_full"])
    out["R_out_A"] = auc["A_full"] - auc["A_lung"];  out["R_out_B"] = auc["B_full"] - auc["B_lung"]   # dış bölgeyi kaldırınca düşüş
    out["R_lung_A"] = auc["A_full"] - auc["A_out"];  out["R_lung_B"] = auc["B_full"] - auc["B_out"]   # akciğeri kaldırınca düşüş
    out["dR_out"] = out["R_out_B"] - out["R_out_A"]
    out["dAUROC"] = auc["B_full"] - auc["A_full"]; out["dAUPRC"] = out["ap_B"] - out["ap_A"]
    return out

rng = np.random.default_rng(123); n = len(yte); boots = []
for _ in range(CFG['n_boot']):
    idx = rng.integers(0, n, n); yt = yte[idx]
    if yt.sum() == 0: continue
    boots.append(stats(yt, {k: v[idx] for k, v in P.items()}))
point = stats(yte, P)
ci = {k: (float(np.percentile([b[k] for b in boots], 2.5)), float(np.percentile([b[k] for b in boots], 97.5))) for k in point}
RESULTS["point"] = {k: float(v) for k, v in point.items()}; RESULTS["ci95"] = ci

def show(k, label=None):
    print(f"{label or k:<34s} {point[k]:+.4f}  [%95 GA {ci[k][0]:+.4f}, {ci[k][1]:+.4f}]")
print("--- AUROC (topluluk, test) ---")
for k in keys: show(k)
print("\n--- AUPRC ---"); show("ap_A", "AUPRC A"); show("ap_B", "AUPRC B")
print("\n--- Ablasyon düşüşleri ---")
for k in ["R_out_A", "R_out_B", "R_lung_A", "R_lung_B"]: show(k)
print("\n--- Karşılaştırmalar (B - A) ---")
for k in ["dR_out", "dAUROC", "dAUPRC"]: show(k)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
names = ["A_full", "A_lung", "A_out", "B_full", "B_lung", "B_out", "C", "D"]
vals = [point[k] for k in names]; lo = [point[k] - ci[k][0] for k in names]; hi = [ci[k][1] - point[k] for k in names]
ax[0].bar(range(len(names)), vals, yerr=[lo, hi], capsize=3); ax[0].axhline(0.5, color="r", ls="--", lw=1)
ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels(names, rotation=45, fontsize=8); ax[0].set_ylabel("AUROC"); ax[0].set_title("Girdi bileşimine göre AUROC")
from sklearn.metrics import roc_curve
for k, lab in [("A_full", "A_raw"), ("B_full", "B_soft")]:
    fpr, tpr, _ = roc_curve(yte, P[k]); ax[1].plot(fpr, tpr, label=f"{lab} (AUROC={point[k]:.3f})")
ax[1].plot([0, 1], [0, 1], "k:", lw=1); ax[1].set_xlabel("1 - özgüllük"); ax[1].set_ylabel("duyarlılık"); ax[1].legend(); ax[1].set_title("ROC (test, topluluk)")
plt.tight_layout(); plt.savefig(os.path.join(CFG['out_dir'], "auroc_ablasyon_roc.png"), dpi=150); plt.show()

#### 7. Grad-CAM ile Dikkat Analizi (H3)

Test kümesindeki **pnömoni pozitif** görüntüler için (en fazla `n_cam`, rastgele) A ve B modellerinin Grad-CAM haritaları çıkarılır (Selvaraju ve ark., ICCV 2017; son evrişim bloğu). **Akciğer içi dikkat oranı** = (harita × akciğer maskesi toplamı) / harita toplamı. Taban çizgisi: maskenin görüntü içindeki alan oranı (dikkat tamamen düzgün dağılsaydı beklenen oran). Oran farkı **eşleştirilmiş Wilcoxon** ile sınanır; tohumlar arası görüntü başına ortalama alınır.

**Uyarı.** B'nin girdisinde dış bölge zaten sönüktür; bu, dikkatin akciğere kaymasını *kısmen yapısal* kılar. Grad-CAM ayrıca bir *kanıt* değil, bir *gösterge*dir (haritalar nedensellik kanıtlamaz, bkz. Adebayo ve ark., NeurIPS 2018, "Sanity checks for saliency maps").

In [ ]:
def gradcam(model, norm, X, L, idx, mode="full", bs=32):
    fill, mean, std = norm
    model.to(device).eval(); layer = cam_layer(model); store = {}
    def fwd(m, i, o):
        store["a"] = o; o.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(fwd); cams = []
    for i in range(0, len(idx), bs):
        j = idx[i:i+bs]
        x = (compose(X[j].to(device), L[j].to(device), mode, fill) - mean) / std
        with torch.enable_grad():
            logit = model(x).squeeze(1); model.zero_grad(); logit.sum().backward()
        a, g = store["a"].detach(), store["g"].detach()
        cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cams.append(cam.cpu())
    h.remove(); model.cpu()
    return torch.cat(cams)

pos_idx = np.flatnonzero(yte == 1)
rs = np.random.default_rng(7)
cam_idx = np.sort(rs.choice(pos_idx, min(CFG['n_cam'], len(pos_idx)), replace=False)) if len(pos_idx) else pos_idx
Lc = LUNG["test"][cam_idx].float()
lung_frac = Lc.mean((1, 2)).numpy()

def in_lung_ratio(cams):
    s = cams.sum((1, 2)); r = (cams * Lc).sum((1, 2)) / s.clamp_min(1e-8)
    r = r.numpy().copy(); r[s.numpy() < 1e-8] = np.nan
    return r

CAMS, RATIO = {}, {}
for name in ["A_raw", "B_soft"]:
    var = EXPS[name]["var"]; per_seed = []; CAMS[name] = []
    for s in CFG['seeds']:
        cams = gradcam(MODELS[(name, s)], NORMS[(name, s)], data_of(var, "test"), LUNG["test"], cam_idx)
        CAMS[name].append(cams); per_seed.append(in_lung_ratio(cams))
    RATIO[name] = np.nanmean(np.stack(per_seed), axis=0)

for name in ["A_raw", "B_soft"]:   # tanı: CAM'i tamamen sıfır (ReLU sonrası) olan görüntüler dışlanır
    zero = int(sum((c_.sum((1, 2)) < 1e-8).sum() for c_ in CAMS[name]))
    print(f"{name}: sıfır-CAM (görüntü x tohum) sayısı = {zero} / {len(cam_idx) * len(CFG['seeds'])}")
    if zero > 0.3 * len(cam_idx) * len(CFG['seeds']):
        print(f"  UYARI: {name} için CAM'lerin %30'undan fazlası sıfır; bu görüntüler dışlandığından H3 yalnızca modelin 'pozitif kanıt' ürettiği örneklere dayanır (seçilim yanlılığı). Sonucu bu kısıtla yorumlayın.")
ok = ~np.isnan(RATIO["A_raw"]) & ~np.isnan(RATIO["B_soft"])
dA, dB = RATIO["A_raw"][ok], RATIO["B_soft"][ok]
RESULTS["cam"] = dict(n=int(ok.sum()), n_candidates=int(len(cam_idx)), lung_frac_median=float(np.median(lung_frac[ok])),
                      ratio_A_median=float(np.median(dA)), ratio_B_median=float(np.median(dB)),
                      diff_median=float(np.median(dB - dA)))
if ok.sum() >= 10 and np.any(dB != dA):
    w, pw_ = wilcoxon(dB, dA); RESULTS["cam"]["wilcoxon_p"] = float(pw_)
else:
    RESULTS["cam"]["wilcoxon_p"] = float("nan")
c = RESULTS["cam"]
print(f"n={c['n']} pozitif görüntü | taban (maske alan oranı) medyan={c['lung_frac_median']:.3f}")
print(f"akciğer içi dikkat oranı medyan: A={c['ratio_A_median']:.3f}  B={c['ratio_B_median']:.3f}  fark(B-A)={c['diff_median']:+.3f}  Wilcoxon p={c['wilcoxon_p']:.3g}")

# örnek görseller (ilk tohum)
show_n = min(4, len(cam_idx)); fig, ax = plt.subplots(3, show_n, figsize=(2.6 * show_n, 7.5))
for c_ in range(show_n):
    i = int(cam_idx[c_])
    ax[0][c_].imshow(XR["test"][i], cmap="gray"); ax[0][c_].set_title("ham", fontsize=8)
    for r_, name in [(1, "A_raw"), (2, "B_soft")]:
        base = (XR if name == "A_raw" else XS)["test"][i]
        cm = CAMS[name][0][c_]; cm = cm / (cm.max() + 1e-8)
        ax[r_][c_].imshow(base, cmap="gray"); ax[r_][c_].imshow(cm, cmap="jet", alpha=0.4)
        ax[r_][c_].contour(LUNG["test"][i].numpy(), levels=[0.5], colors="w", linewidths=0.6)
        ax[r_][c_].set_title(f"{name} Grad-CAM", fontsize=8)
    for r_ in range(3): ax[r_][c_].axis("off")
plt.tight_layout(); plt.savefig(os.path.join(CFG['out_dir'], "gradcam_ornekler.png"), dpi=150); plt.show()

#### 8. Fallback Alt Grup Analizi

`step_02`'te maskesi bozuk bulunan (alanı eşiğin altında kalan) test görüntülerinde B modeli **orijinal görüntüyü** görmüştür. Bu alt grupta A ve B'nin davranışı ayrı raporlanır. **Uyarı:** alt grupta pozitif sayısı çok azsa (< 10) sonuç yorumlanmamalıdır.

In [ ]:
fl = FAIL["test"]
RESULTS["fallback"] = {"n_fail": int(fl.sum()), "n_fail_pos": int(yte[fl].sum()), "n_ok_pos": int(yte[~fl].sum())}
print(f"fallback grubu: {fl.sum()} görüntü, {int(yte[fl].sum())} pozitif | diğer: {(~fl).sum()} görüntü, {int(yte[~fl].sum())} pozitif")
for lab, mask in [("fallback", fl), ("maskeli", ~fl)]:
    if yte[mask].sum() >= 10 and yte[mask].sum() < mask.sum():
        a_, b_ = roc_auc_score(yte[mask], P["A_full"][mask]), roc_auc_score(yte[mask], P["B_full"][mask])
        RESULTS["fallback"][lab] = dict(auroc_A=float(a_), auroc_B=float(b_))
        print(f"  {lab}: AUROC A={a_:.3f}  B={b_:.3f}")
    else:
        print(f"  {lab}: pozitif sayısı yetersiz, AUROC hesaplanmadı")

#### 9. Hipotezlerin Önceden Tanımlı Ölçütlerle Değerlendirilmesi

Aşağıdaki hücre, bölüm başındaki ölçütleri **olduğu gibi** uygular ve her hipotez için *desteklendi / desteklenmedi / zayıf kanıt* çıktısı verir. Sonuç beklentinizin tersine çıkarsa bu **bir bulgudur** ve sunumda raporlanmalıdır; ölçütleri sonradan değiştirmeyin.

In [ ]:
V = {}
lo_c, hi_c = ci["C"]
V["H1"] = "DESTEKLENDİ" if lo_c > 0.55 else ("ZAYIF KANIT" if lo_c > 0.50 else "DESTEKLENMEDİ")
V["H2"] = "DESTEKLENDİ" if ci["dR_out"][1] < 0 else "DESTEKLENMEDİ"
c = RESULTS["cam"]
V["H3"] = "DESTEKLENDİ" if (c["wilcoxon_p"] < 0.05 and c["diff_median"] > 0) else "DESTEKLENMEDİ"
V["H4"] = "AŞAĞI OLMAMA SAĞLANDI" if ci["dAUROC"][0] > -0.02 else "AŞAĞI OLMAMA SAĞLANMADI"
print(f"H1  C_outside AUROC={point['C']:.3f}  GA alt sınır={lo_c:.3f}  -> {V['H1']}")
print(f"H2  ΔR_dış={point['dR_out']:+.3f}  GA=[{ci['dR_out'][0]:+.3f}, {ci['dR_out'][1]:+.3f}]  -> {V['H2']}")
print(f"H3  medyan fark={c['diff_median']:+.3f}  Wilcoxon p={c['wilcoxon_p']:.3g}  -> {V['H3']}")
print(f"H4  ΔAUROC={point['dAUROC']:+.4f}  GA=[{ci['dAUROC'][0]:+.4f}, {ci['dAUROC'][1]:+.4f}]  -> {V['H4']}")
RESULTS["verdict"] = V
RESULTS["times_min"] = {f"{k[0]}_s{k[1]}": round(v / 60, 2) for k, v in TIMES.items()}
if CFG['quick']: print("\nUYARI: quick=True; bu sonuçlar yalnızca kod denemesidir, raporlanmaz.")

def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
with open(os.path.join(CFG['out_dir'], "results_step04.json"), "w", encoding="utf-8") as f:
    json.dump(_clean(RESULTS), f, ensure_ascii=False, indent=2)
print("results_step04.json kaydedildi:", CFG['out_dir'])

#### 10. Sonuçları Yorumlama Kılavuzu (olası durumlar)

| Gözlem | Olası yorum | Dikkat |
|---|---|---|
| **H1 desteklendi** (C_outside AUROC > 0,5) | Akciğer dışında pnömoniyle ilişkili bilgi var. | Bu bilgi *kısayol* olabilir (cihaz, çekim koşulu) ama gerçek bir klinik ilişki de olabilir (ör. kalp gölgesi, plevral/diyafram bulguları, hasta ağırlığı; maske sınırı akciğeri tam kapsamıyorsa akciğer kenar dokusu "dış" sayılabilir). Kısayol **kanıtı değil, kısayol potansiyelidir.** |
| **H1 desteklenmedi** | Bu çözünürlükte/etiketle dış sinyal zayıf; kısayol riski düşük olabilir. | 128×128'de küçük cihaz izleri kaybolmuş olabilir; "yok" değil "ölçülemedi" olabilir. |
| **H2/H3 desteklendi** | B, kararını daha çok akciğere dayandırıyor. | B'nin dışı zaten sönük (yapısal etki); H1'le birlikte oku. |
| **H4: aşağı olmama sağlandı** | Soft mask performansı anlamlı düşürmeden dikkati akciğere çekiyor. | Test'te ~242 pozitif → GA geniş; "fark yok" ≠ "eşdeğer". |
| **H4 sağlanmadı (B daha kötü)** | Dış bağlam hastalık sinyaline de katkı sağlıyor olabilir (ör. hasta durumu) veya maske hataları bilgi kaybettiriyor. | Maske kalitesi ve fallback oranına bak. |
| **B ≥ A ama H1 desteklenmedi** | Soft mask iyileştirmesi kısayol azaltmadan değil, düzenlileştirme/gürültü bastırma etkisinden gelebilir. | Mekanizma iddiası yapma. |

#### 11. Bu Çalışmanın Genel Sınırlılıkları

1. **Dış geçerlilik yok:** Tek veri kaynağı; başka hastane/cihazda performans bilinmez. Kısayolun *azaldığı* yalnızca dolaylı kanıtlarla (H1–H3) savunulabilir, doğrudan kanıt için dış doğrulama gerekir.
2. **Nedensellik yok:** Ablasyon ve Grad-CAM bağıntısal göstergelerdir.
3. **Maske kalitesi ölçülmedi (referans maske yok):** Hatalı maskeler sonuçları sistematik biçimde etkileyebilir; pnömonili akciğerlerde segmentasyon hatası ayrıca incelenmelidir.
4. **Hiperparametre ablasyonu yok:** `ALPHA`, `DILATE_PX`, `BLUR_PX`, eşik ve eğitim hiperparametreleri taranmadı.
5. **Tek mimari ve sınırlı tohum:** ResNet-18, A/B için 3, C/D için 1 tohum. Farklı mimarilerde sonuç değişebilir.
6. **Etiket gürültüsü ve 128×128 çözünürlük:** Mutlak performansın üst sınırını belirler.
7. **Çoklu karşılaştırma:** Dört hipotez sınandı; düzeltme uygulanmadı, ölçütler önceden sabitlenerek serbestlik derecesi sınırlandı.
8. **Klinik kullanım için uygun değildir.**